In [1]:
# Importing required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline
import seaborn as sns
import scipy.stats as si
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from scipy import stats as st
from scipy.optimize import least_squares
from tabulate import tabulate
import warnings
warnings.filterwarnings("ignore")
from scipy.stats.mstats import gmean
import statsmodels.formula.api as smf
import itertools
from itertools import product

c:\Users\YeonChan Kang\anaconda3\envs\fdb\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Data

In [2]:
oa_factor = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/oa_data/PredictorLSretWide.csv")

In [20]:
market_ret = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/oa_data/market_return.csv").dropna()

In [33]:
market_ret.reset_index(drop=True, inplace=True)

In [35]:
df = pd.concat([oa_factor, market_ret['vwretd']], axis=1)

In [36]:
df

,date,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,AgeIPO,...,roaq,sfe,sinAlgo,skew1,std_turn,tang,zerotrade,zerotradeAlt1,zerotradeAlt12,vwretd
0,1926-01-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-13.686393,NaN,NaN,NaN,NaN,NaN,NaN,0.000561
1,1926-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-5.135248,NaN,NaN,NaN,NaN,NaN,NaN,-0.033046
2,1926-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-4.832534,NaN,NaN,NaN,NaN,NaN,NaN,-0.064002
3,1926-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-4.440934,NaN,NaN,NaN,NaN,NaN,NaN,0.037029
4,1926-05-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.483265,NaN,NaN,NaN,NaN,NaN,NaN,0.012095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1159,2022-08-31,1.019538,-2.412224,-1.913597,1.928721,2.741776,NaN,NaN,1.347350,-4.050457,...,-11.692944,-8.326179,-4.467901,0.216663,-5.978392,8.577257,-2.591933,-2.451913,-1.467892,-0.036233
1160,2022-09-30,-2.888651,2.895650,-0.492878,1.546341,-5.482939,NaN,NaN,-6.515619,-2.104168,...,2.106418,2.906625,-3.030688,0.105947,8.239848,0.485810,10.430349,6.948029,4.607782,-0.091323
1161,2022-10-31,3.384184,-3.041480,-0.437972,1.439492,14.154554,NaN,NaN,-2.241708,5.847384,...,13.064743,9.269617,-0.339419,-0.638394,-4.242074,-8.276771,-4.179820,-3.358313,-5.818550,0.077394
1162,2022-11-30,-0.060910,2.179653,0.734355,-1.934128,-0.503022,NaN,NaN,-0.914064,5.350460,...,11.645717,14.288103,-0.150439,2.255298,5.601910,-5.353664,-6.738538,-2.475098,-2.581228,0.052354


# Factor Momentum EveryWhere

# Return

In [4]:
def plot_and_save_factor_returns(df, time_col, save_plots=False, output_dir='factor_returns', img_show=None):
    """
    Plot and save the factor returns for each factor in a DataFrame.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing factor returns.
    save_plots (bool): Whether to save the plots as images. Default is False.
    output_dir (str): The directory to save the plots. Default is 'factor_returns'.
    img_show (bool): Whether to display the plots. Default is None.
    """
    # Create output directory if it does not exist
    if save_plots and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    df = df.drop(columns=[time_col]) # time을 X 축으로 어떻게?

    # Plot and save each factor
    for column in df.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(df.index, df[column], label=column)
        plt.title(f'Returns for {column}')
        plt.xlabel('Date')
        plt.ylabel('Return')
        plt.legend()
        plt.grid(True)
        
        plt.xticks(rotation=45)
        
        if save_plots:
            plt.savefig(os.path.join(output_dir, f'{column}_returns.png'))
        
        if img_show:
            plt.show()

In [12]:
ch_col = oa_factor.columns[1:].to_list()

In [9]:
re_save = "C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/image/oa/ret"

# Alpha

In [40]:
import wrds
import pandas as pd

# Connect to WRDS
conn = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [47]:
query = """
    SELECT mcaldt, tfz_mth.tmyld
    FROM crsp.tfz_mth
    WHERE mcaldt >= '1926-01-01' AND mcaldt <= '2023-12-31'
"""
t_bill_data = conn.raw_sql(query)

# Convert to DataFrame and set the date as index
t_bill_df = pd.DataFrame(t_bill_data)
t_bill_df['mcaldt'] = pd.to_datetime(t_bill_df['mcaldt'])
t_bill_df.set_index('mcaldt', inplace=True)

# Display the first few rows
print(t_bill_df.head())
print(t_bill_df.tail())


               tmyld
mcaldt              
1960-10-31  0.000076
1960-11-30  0.000079
1960-12-30  0.000067
1961-01-31  0.000068
1961-02-28  0.000074
               tmyld
mcaldt              
2023-12-29  0.000129
2023-12-29  0.000115
2023-12-29  0.000109
2023-12-29  0.000104
2023-12-29  0.000105


In [48]:
t_bill_df

,tmyld
mcaldt,
1960-10-31,0.000076
1960-11-30,0.000079
1960-12-30,0.000067
1961-01-31,0.000068
1961-02-28,0.000074
...,...
2023-12-29,0.000129
2023-12-29,0.000115
2023-12-29,0.000109


In [ ]:
market_return = df['Market']  # Replace with actual market return column
risk_free_rate = df['Rf']  # Replace with actual risk-free rate column
smb = df['SMB']  # Replace with actual SMB column
hml = df['HML']  # Replace with actual HML column
rmw = df['RMW']  # Replace with actual RMW column
cma = df['CMA']  # Replace with actual CMA column
momentum = df['MOM']  # Replace with actual MOM column